In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# 結果の可視化

<details>
<summary><b>Package versions</b></summary>

このページのコードは、以下の要件を使用して開発されました。
これらのバージョン以降を使用することを推奨します。

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

## ヒストグラムのプロット
`plot_histogram` 関数は、QPU上で量子回路をサンプリングした結果を可視化します。

> **Note:** この関数は `matplotlib.Figure` オブジェクトを返します。コードセルの最後の行がこれらのオブジェクトを出力する場合、Jupyterノートブックはセルの下に表示します。他の環境やスクリプトでこれらの関数を呼び出す場合は、明示的に出力を表示または保存する必要があります。

2つのオプションがあります：
- 返されたオブジェクトに対して `.show()` を呼び出し、新しいウィンドウで画像を開きます（設定されたmatplotlibバックエンドがインタラクティブであることが前提です）。
- `.savefig("out.png")` を呼び出し、現在の作業ディレクトリの `out.png` に図を保存します。`savefig()` メソッドはパスを受け取るため、出力を保存する場所とファイル名を調整できます。例えば、`plot_state_city(psi).savefig("out.png")` のように使用します。

例として、2量子ビットのベル状態を作成します：

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler import generate_preset_pass_manager

from qiskit.circuit import QuantumCircuit
from qiskit.visualization import plot_histogram

service = QiskitRuntimeService()

backend = service.least_busy(simulator=False, operational=True)

In [2]:
# Quantum circuit to make a Bell state
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(bell)

# execute the quantum circuit
sampler = Sampler(backend)
job = sampler.run([isa_circuit])
result = job.result()

print(result)

PrimitiveResult([SamplerPubResult(data=DataBin(meas=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}})], metadata={'execution': {'execution_spans': ExecutionSpans([DoubleSliceSpan(<start='2026-01-15 07:11:30', stop='2026-01-15 07:11:32', size=4096>)])}, 'version': 2})


In [3]:
plot_histogram(result[0].data.meas.get_counts())

<Image src="../docs/images/guides/visualize-results/extracted-outputs/57d8053e-d030-460d-9c1f-772e53b1a49b-0.svg" alt="Output of the previous code cell" />

### Options when plotting a histogram

Use the following options for `plot_histogram` to adjust the output graph.

* `legend`: Provides a label for the executions. It takes a list of strings used to label each execution's results. This is mostly useful when plotting multiple execution results in the same histogram
* `sort`: Adjusts the order of the bars in the histogram. It can be set to either ascending order with `asc` or descending order with `desc`
* `number_to_keep`: Takes an integer for the number of terms to show. The rest are grouped together in a single bar called "rest"
* `color`: Adjusts the color of the bars; takes a string or a list of strings for the colors to use for the bars for each execution
* `bar_labels`: Adjusts whether labels are printed above the bars
* `figsize`: Takes a tuple of the size in inches to make the output figure

In [4]:
# Execute two-qubit Bell state again
sampler.options.default_shots = 1000

job = sampler.run([isa_circuit])
second_result = job.result()

# Plot results with custom options
plot_histogram(
    [
        result[0].data.meas.get_counts(),
        second_result[0].data.meas.get_counts(),
    ],
    legend=["first", "second"],
    sort="desc",
    figsize=(15, 12),
    color=["orange", "black"],
    bar_labels=False,
)

<Image src="../docs/images/guides/visualize-results/extracted-outputs/bd70e13f-5c52-42fb-8dde-980b15e3604a-0.svg" alt="Output of the previous code cell" />

![前のコードセルの出力](../docs/images/guides/visualize-results/extracted-outputs/57d8053e-d030-460d-9c1f-772e53b1a49b-0.svg)

### ヒストグラムをプロットする際のオプション
`plot_histogram` の以下のオプションを使用して、出力グラフを調整できます。

* `legend`: 実行結果にラベルを付けます。各実行結果にラベルを付けるための文字列のリストを受け取ります。主に同じヒストグラムに複数の実行結果をプロットする場合に便利です。
* `sort`: ヒストグラムのバーの順序を調整します。`asc` で昇順、`desc` で降順に設定できます。
* `number_to_keep`: 表示する項目数を整数で指定します。残りは「rest」という単一のバーにまとめられます。
* `color`: バーの色を調整します。各実行のバーに使用する色を文字列または文字列のリストで指定します。
* `bar_labels`: バーの上にラベルを表示するかどうかを調整します。
* `figsize`: 出力図のサイズ（インチ単位）のタプルを受け取ります。

In [5]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.transpiler import generate_preset_pass_manager
from matplotlib import pyplot as plt

# Simple estimation experiment to create results
qc = QuantumCircuit(2)
qc.h(0)
qc.crx(1.5, 0, 1)

observables_labels = ["ZZ", "XX", "YZ", "ZY", "XY", "XZ", "ZX"]
observables = [SparsePauliOp(label) for label in observables_labels]

service = QiskitRuntimeService()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
isa_observables = [
    operator.apply_layout(isa_circuit.layout) for operator in observables
]

# Reshape observable array for broadcasting
reshaped_ops = np.fromiter(isa_observables, dtype=object)
reshaped_ops = reshaped_ops.reshape((7, 1))

estimator = Estimator(backend)
job = estimator.run([(isa_circuit, reshaped_ops)])
result = job.result()[0]
exp_val = job.result()[0].data.evs
print(result)

# Since the result array is structured as a 2D array where each element is a
# list containing a single value, you need to flatten the array.

# Plot using Matplotlib
plt.bar(observables_labels, exp_val.flatten())

PubResult(data=DataBin(evs=np.ndarray(<shape=(7, 1), dtype=float64>), stds=np.ndarray(<shape=(7, 1), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(7, 1), dtype=float64>), shape=(7, 1)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})


<BarContainer object of 7 artists>

<Image src="../docs/images/guides/visualize-results/extracted-outputs/17c9893a-d1bf-4726-b444-6dce1d56805f-2.svg" alt="Output of the previous code cell" />

![前のコードセルの出力](../docs/images/guides/visualize-results/extracted-outputs/bd70e13f-5c52-42fb-8dde-980b15e3604a-0.svg)

## Estimatorの結果をプロットする
Qiskitには Estimator の結果をプロットするための組み込み関数はありませんが、Matplotlibの [`bar` プロット](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.bar.html) を使って手軽に可視化できます。

デモとして、以下のセルでは量子状態における7種類のオブザーバブルの期待値を推定します。

In [6]:
standard_error = job.result()[0].data.stds

_, ax = plt.subplots()
ax.bar(
    observables_labels,
    exp_val.flatten(),
    yerr=standard_error.flatten(),
    capsize=2,
)
ax.set_title("Expectation values (with standard errors)")

Text(0.5, 1.0, 'Expectation values (with standard errors)')

<Image src="../docs/images/guides/visualize-results/extracted-outputs/4eb79f4b-36b5-4797-a1a0-67d881d46ca4-1.svg" alt="Output of the previous code cell" />